# IMPORTS

In [ ]:
# install
!pip install -q efficientnet-pytorch
!pip install -q monai
!pip install -q timm
!pip install -q ranger21

# Core
import os
import cv2
import io
import base64
import json
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict
import shutil
import random
import warnings
warnings.filterwarnings('ignore')

# Torch
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms, models

# monai
import monai
from monai.losses import DiceCELoss, DiceFocalLoss, HausdorffDTLoss
from monai.metrics import DiceMetric, compute_hausdorff_distance, compute_dice
from monai.utils import MetricReduction

# resnest
import timm

# SKLearn
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, accuracy_score

# Progress bar
from tqdm import tqdm

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Jupyter widgets
from IPython.display import HTML, display
import ipywidgets as widgets

# optimizer
from ranger21 import Ranger21

# LOADING DATASET

In [2]:
def load_patient_data(csv_path, images_dir):
    df = pd.read_excel(csv_path)
    patients_dict = defaultdict(list)
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Caricamento dati"):
        img_name = row['US']
        mask_name = row['MASK']
        label = row['LABEL']
        
        img_stem = Path(img_name).stem
        parts = img_stem.split('_')
        patient_id = parts[0]
        
        img_path = str(Path(images_dir) / img_name)
        mask_path = str(Path(images_dir) / mask_name)
        
        pair = {
            'img_path': img_path,
            'mask_path': mask_path,
            'img_name': img_name,
            'label': label
        }
        
        patients_dict[patient_id].append(pair)
    patients = []
    for pid, samples in sorted(patients_dict.items(), key=lambda x: int(x[0])):
        labels = [s['label'] for s in samples]
        label_counts = np.bincount(labels)
        major_label = int(np.argmax(label_counts))
        
        patients.append({
            'patient_id': pid,
            'samples': samples,
            'major_label': major_label
        })
    
    return patients

In [3]:
INPUT_DIRECTORY = '/kaggle/input/breast-lesions/'
TRAINING_DIRECTORY = INPUT_DIRECTORY + 'train/'
IMG_DIRECTORY = TRAINING_DIRECTORY + 'images'
LABELS_DIRECTORY = TRAINING_DIRECTORY + 'train_metadata.xlsx'

train_data = load_patient_data(LABELS_DIRECTORY, IMG_DIRECTORY)
train_labels = pd.read_excel(LABELS_DIRECTORY)

print(f"\n✅ Loaded {len(train_data)} patients.")

Caricamento dati: 100%|██████████| 1195/1195 [00:00<00:00, 10994.91it/s]


✅ Loaded 774 patients.


In [4]:
label_map = {
    0: '0 - BENIGN',
    1: '1 - MALIGNANT',
    2: '2 - NORMAL'
}

# PRE-PROCESSING

## Crop - Square Pad
- Crop dell'Area Attiva: Elimina le righe e colonne nere (o quasi) ai bordi
- Square Padding (Letterboxing)
    - Calcola il lato maggiore del ritaglio
    - Aggiungi padding nero al lato più corto per rendere l'immagine quadrata.
- Resize Finale a output size

In [5]:
def square_pad_resize(img, target_size, pad_value=0):
    h, w = img.shape[:2]
    
    if h == w and h == target_size:
        return img

    max_dim = max(h, w)

    
    delta_w = max_dim - w
    delta_h = max_dim - h
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)
    

    img_padded = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=pad_value)
    

    img_resized = cv2.resize(img_padded, (target_size, target_size), interpolation=cv2.INTER_CUBIC)
    
    return img_resized

## Inpainting

In [6]:
def has_colors(img,
            sat_thresh=40,
            delta_thresh=20,
            min_region_area=20,
            max_region_area_ratio=0.1,
            max_aspect_ratio=5,
            mean_delta_thresh=15,
            dilate_iters=1,
            erode_iters=1):


    # --- RGB disagreement ---
    b, g, r = cv2.split(img)
    r = r.astype(np.int16)
    g = g.astype(np.int16)
    b = b.astype(np.int16)

    delta = np.maximum.reduce([np.abs(r - g), np.abs(r - b), np.abs(g - b)])
    mask_rgb = delta > delta_thresh

    # --- Saturation check ---
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask_sat = hsv[:, :, 1] > sat_thresh

    # --- Combine pixel-level masks ---
    mask = np.logical_and(mask_rgb, mask_sat).astype(np.uint8) * 255
    if np.sum(mask) == 0:
        return False, mask

    # --- Morphology: controlled directional dilation + closing ---
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 1))
    kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 3))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

    mask = cv2.dilate(mask, kernel_h, iterations=dilate_iters)
    mask = cv2.dilate(mask, kernel_v, iterations=dilate_iters)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    # if erode_iters > 0:
    #    mask = cv2.erode(mask, kernel_close, iterations=erode_iters)

    # --- Connected component filtering ---
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    h, w = mask.shape
    img_area = h * w

    filtered = np.zeros_like(mask)
    rect = np.zeros_like(mask)
    detected = False

    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        x, y, width, height = stats[i, cv2.CC_STAT_LEFT], stats[i, cv2.CC_STAT_TOP], stats[i, cv2.CC_STAT_WIDTH], stats[i, cv2.CC_STAT_HEIGHT]
        aspect_ratio = max(width / (height + 1e-5), height / (width + 1e-5))

        # Extract region pixels
        region_mask = labels == i
        mean_delta = delta[region_mask].mean()
        std_delta = delta[region_mask].std()
        mean_sat = hsv[:, :, 1][region_mask].mean()
        std_sat = hsv[:, :, 1][region_mask].std()
        

        if (min_region_area < area < max_region_area_ratio * img_area):
            filtered[region_mask] = 255
            detected = True
    
        elif ((aspect_ratio > max_aspect_ratio) and 
            (mean_delta > mean_delta_thresh) and std_delta < 1 and 
            mean_sat > 25 and std_sat < 1):
            rect[region_mask] = 255
            detected = True
            
    # --- Morphology: controlled directional dilation + closing ---
    kernel_h = cv2.getStructuringElement(cv2.MORPH_RECT, (200, 1))
    kernel_v = cv2.getStructuringElement(cv2.MORPH_RECT, (1, 200))
    kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))

    rect = cv2.dilate(rect, kernel_h, iterations=dilate_iters)
    rect = cv2.dilate(rect, kernel_v, iterations=dilate_iters)
    rect = cv2.morphologyEx(rect, cv2.MORPH_CLOSE, kernel_close)

    kernel_corner = cv2.getStructuringElement(cv2.MORPH_RECT, (10, 10))
    rect = cv2.morphologyEx(rect, cv2.MORPH_CLOSE, kernel_corner)

    if erode_iters > 0:
        rect = cv2.erode(rect, kernel_close, iterations=erode_iters)
    
    filtered = cv2.bitwise_or(filtered, rect)

    return detected, filtered

In [7]:
def remove_ink(img, ink_mask):

    # OpenCV inpainting expects 8-bit single channel mask
    mask = ink_mask
    

    img_inpainted = cv2.inpaint(img, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    
    return img_inpainted

In [8]:
def inpaint(img):

    hasColors, color_mask = has_colors(img)
    
    if hasColors:
        img_clean = remove_ink(img, color_mask)
        return img_clean
    else:
        return img

## Noise Filter

In [9]:
# median filtering
def apply_median_filter(image, kernel_size=5):
    denoised = cv2.medianBlur(image.astype(np.uint8), kernel_size)
    return denoised

## Normalization

In [10]:
def apply_clahe(image, clip_limit=2.0, tile_grid_size=(8, 8)):

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(image)

In [11]:
def apply_zscore_normalization(image):

    image = image.astype(np.float32)
    mean = np.mean(image)
    std = np.std(image)
    
    if std == 0:
        return image - mean
    
    return (image - mean) / std

# TRAINING SETUP

## Configuration

In [12]:
class Config:
    # Training
    NUM_EPOCHS = 50
    BATCH_SIZE = 16
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    
    # Data
    CONTEXT_FACTOR = 2.0
    NUM_WORKERS = 2
    OUTPUT_SIZE = 384
    
    # Model
    NUM_CLASSES = 3
    INPUT_CHANNELS = 1
    
    # Class weights per Weighted Cross Entropy (adjust based on imbalance)
    CLASS_WEIGHTS = torch.tensor([0.8, 1.5, 1.0])  # Benign, Malignant, Normal
    
    # Device
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Data Augmentation

In [13]:
class RandomChannelPermutation:
    def __init__(self, p=0.5):
        self.p = p

    def __call__(self, img):
        if torch.rand(1) < self.p:
            # Genera permutazione casuale di (0, 1, 2)
            perm = torch.randperm(3)
            img = img[perm, :, :]
        return img

In [14]:
class UltrasoundAugmentation:
    def __init__(self, p=0.5):
        self.p = p
    
    def __call__(self, img_tensor):
        flip_h = random.random() < 0.5
        flip_v = random.random() < 0.5
        apply_rotation = random.random() < self.p
        rotation_angle = random.uniform(-15, 15) if apply_rotation else 0
        

        
        # Horizontal Flip
        if flip_h:
            img_tensor = torch.flip(img_tensor, dims=[2])
        
        # Vertical Flip
        if flip_v:
            img_tensor = torch.flip(img_tensor, dims=[1])
        
        # Random Rotation
        if apply_rotation:
            pil_img = transforms.ToPILImage()(img_tensor)
            pil_img = transforms.functional.rotate(pil_img, rotation_angle, fill=0)
            img_tensor = transforms.ToTensor()(pil_img)
            
        # Elastic Transform
        if random.random() < self.p:
            # Alpha: intensità della deformazione (scaling dello spostamento)
            # Sigma: morbidezza della deformazione (Gaussian smoothing)
            alpha_val = random.uniform(40.0, 60.0)
            sigma_val = random.uniform(4.0, 6.0)
            
            # Richiede torchvision >= 0.11.0
            # fill=0 assume sfondo nero (tipico US)
            elastic = transforms.ElasticTransform(alpha=alpha_val, sigma=sigma_val, fill=0)
            img_tensor = elastic(img_tensor)
        
        
        # ===== AUGMENTATION DI INTENSITÀ (SOLO immagine) =====
        # Brightness & Contrast vengono applicati SOLO all'immagine
        if random.random() < self.p:
            brightness_factor = random.uniform(0.9, 1.1)
            img_tensor = transforms.functional.adjust_brightness(
                transforms.ToPILImage()(img_tensor), brightness_factor
            )
            img_tensor = transforms.ToTensor()(img_tensor)

        return img_tensor

## Dataset

In [15]:
class ClassificationDataset(Dataset):

    
    def __init__(self, patients, is_training=True, target_size=Config.OUTPUT_SIZE): 
        self.patients = patients
        self.is_training = is_training
        self.target_size = target_size 
        self.augmentation = UltrasoundAugmentation(p=0.5) if is_training else None
        
        # Flatten dei campioni
        self.all_samples = []
        for patient in patients:
            for sample in patient['samples']:
                self.all_samples.append(sample)
    
    def __len__(self):
        return len(self.all_samples)
    
    def __getitem__(self, idx):
        sample = self.all_samples[idx]
        img_path = sample['img_path']
        label = sample['label']

        # 1. Caricamento
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError(f"Immagine non trovata: {img_path}")

        # 2. Inpainting
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = inpaint(img) 
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)

        # 3. Square Padding + Resize
        img = square_pad_resize(img, target_size=self.target_size)
        
        # 4. Enhancement
        img = apply_median_filter(img)
        img = apply_clahe(img)
        
        # 5. Normalizzazione Z-Score
        img = apply_zscore_normalization(img)
        
        # 6. Conversione a Tensor
        img_tensor = torch.from_numpy(img).float().unsqueeze(0)  # Shape: (1, 384, 384)
        
        # 7. Augmentation
        if self.augmentation is not None:
            img_tensor = self.augmentation(img_tensor)
        
        label_tensor = torch.tensor(label, dtype=torch.long)
        
        return img_tensor, label_tensor

## Data Loaders

In [16]:
def create_data_loaders(train_patients, val_patients, batch_size=32, 
                        context_factor=2.0, num_workers=0):
    train_dataset = ClassificationDataset(
        train_patients, 
        is_training=True
    )
    
    val_dataset = ClassificationDataset(
        val_patients, 
        is_training=False
    )
    
    # Crea dataloader
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers,
        pin_memory=True,
        drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=num_workers,
        pin_memory=True,
        drop_last=False
    )
    
    return train_loader, val_loader

## Model

In [17]:
class GlobalResNeSt50(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):

        super().__init__()
        
        self.model = timm.create_model('resnest50d', pretrained=pretrained)
        
        # --- 1. Adattamento Input: da 3 canali (RGB) a 1 canale (Grayscale) ---
        original_conv = self.model.conv1[0]
        
        new_conv = nn.Conv2d(
            in_channels=1, 
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=original_conv.bias is not None
        )
        
        with torch.no_grad():
            new_conv.weight[:] = torch.sum(original_conv.weight, dim=1, keepdim=True)
            
        self.model.conv1[0] = new_conv
        
        # --- 2. Adattamento Output: Testata di Classificazione ---
        n_features = self.model.fc.in_features
        self.model.fc = nn.Linear(n_features, num_classes)
        
    def forward(self, x):
        return self.model(x)

## Loss

In [18]:
def get_class_weights(labels, device):
    weights = compute_class_weight(
        class_weight='balanced', 
        classes=np.unique(labels), 
        y=labels
    )
    return torch.tensor(weights, dtype=torch.float32).to(device)

## K-Fold Stratified CV

In [19]:
def stratified_kfold_split(patients, n_splits=5, random_state=42):
    major_labels = np.array([p['major_label'] for p in patients])
    
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for train_idx, val_idx in skf.split(np.arange(len(patients)), major_labels):
        train_patients = [patients[i] for i in train_idx]
        val_patients = [patients[i] for i in val_idx]
        
        yield train_patients, val_patients

# TRAINING LOOP

## Manager

In [20]:
class ExperimentManager:
    
    def __init__(self, base_dir='experiment_results'):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)
        self.fold_metrics = []
        
    
    def save_model(self, model, fold_idx, filename='best_model.pth'):

        save_path = self.base_dir / f'fold_{fold_idx}'
        save_path.mkdir(exist_ok=True)
        torch.save(model.state_dict(), save_path / filename)
        
    def save_history(self, history, fold_idx):

        df = pd.DataFrame(history)
        save_path = self.base_dir / f'fold_{fold_idx}'
        save_path.mkdir(exist_ok=True)
        df.to_csv(save_path / 'history.csv', index=False)
        

        self._plot_save_history(df, fold_idx, save_path)

    def save_confusion_matrix(self, true_labels, pred_labels, fold_idx, classes=['Benign', 'Malignant', 'Normal']):

        cm = confusion_matrix(true_labels, pred_labels)
        
        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                    xticklabels=classes, yticklabels=classes)
        plt.title(f'Confusion Matrix - Fold {fold_idx}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        
        save_path = self.base_dir / f'fold_{fold_idx}'
        save_path.mkdir(exist_ok=True)
        plt.savefig(save_path / 'confusion_matrix.png')
        plt.close()
        
    
    def log_fold_metrics(self, fold_idx, true_labels, pred_labels, best_val_loss):

        bal_acc = balanced_accuracy_score(true_labels, pred_labels)
        acc = accuracy_score(true_labels, pred_labels)
        
        metrics = {
            'fold': fold_idx,
            'balanced_accuracy': bal_acc,
            'accuracy': acc,
            'best_val_loss': best_val_loss
        }
        self.fold_metrics.append(metrics)
        

        save_path = self.base_dir / f'fold_{fold_idx}'
        with open(save_path / 'metrics.json', 'w') as f:
            json.dump(metrics, f, indent=4)
            
    def save_final_summary(self):

        df = pd.DataFrame(self.fold_metrics)
        df.to_csv(self.base_dir / 'final_summary.csv', index=False)
        

        print("\n=== FINAL RESULTS ACROSS FOLDS ===")
        print(df.describe().loc[['mean', 'std']])
        
    
    def _plot_save_history(self, df, fold_idx, save_path):

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        # Loss
        axes[0].plot(df['train_loss'], label='Train')
        axes[0].plot(df['val_loss'], label='Val')
        axes[0].set_title(f'Fold {fold_idx} Loss')
        axes[0].legend()
        
        # Balanced Accuracy
        axes[1].plot(df['train_bal_acc'], label='Train')
        axes[1].plot(df['val_bal_acc'], label='Val')
        axes[1].set_title(f'Fold {fold_idx} Bal. Acc.')
        axes[1].legend()
        
        plt.savefig(save_path / 'training_history.png')
        plt.close()

## Functions

In [21]:
def train_epoch(model, train_loader, optimizer, criterion, device, scaler, 
                accumulation_steps=1, max_grad_norm=1.0):

    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    

    optimizer.zero_grad()
    

    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        
        # 1. Forward pass con Mixed Precision
        with torch.amp.autocast('cuda'):
            logits = model(images)
            loss = criterion(logits, labels)
            loss = loss / accumulation_steps
        
        # 2. Backward pass (Scaled)
        scaler.scale(loss).backward()
        
        # 3. Step dell'ottimizzatore (solo ogni accumulation_steps)
        if (batch_idx + 1) % accumulation_steps == 0:
            # Unscale prima del clipping
            scaler.unscale_(optimizer)
            
            # Gradient Clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            
            # Optimizer step & Scaler update
            scaler.step(optimizer)
            scaler.update()
            
            # Zero grad
            optimizer.zero_grad()
        
        # Metriche

        current_loss = loss.item() * accumulation_steps
        total_loss += current_loss
        
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())
        
    avg_loss = total_loss / len(train_loader)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    
    return avg_loss, balanced_acc


def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            
            with torch.amp.autocast('cuda'):
                logits = model(images)
                loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    avg_loss = total_loss / len(val_loader)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)
    
    return avg_loss, balanced_acc, accuracy, all_preds, all_labels


def plot_training_results(history):
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training Loss')
    axes[0].legend()
    axes[0].grid(True)
    
    # Balanced Accuracy
    axes[1].plot(history['train_bal_acc'], label='Train Balanced Acc', marker='o')
    axes[1].plot(history['val_bal_acc'], label='Val Balanced Acc', marker='s')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Balanced Accuracy')
    axes[1].set_title('Balanced Accuracy')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()


def plot_confusion_matrix(all_preds, all_labels, title="Confusion Matrix"):
    label_names = {0: 'Benign', 1: 'Malignant', 2: 'Normal'}
    
    cm = confusion_matrix(all_labels, all_preds)
    accuracy = accuracy_score(all_labels, all_preds)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=[label_names[i] for i in range(3)],
                yticklabels=[label_names[i] for i in range(3)])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(f'{title}\nAccuracy: {accuracy:.4f}')
    plt.tight_layout()
    plt.show()

## Actual Training

In [22]:
# --- CONFIGURAZIONE ---
N_FOLDS = 5
ACCUMULATION_STEPS = 1
MAX_GRAD_NORM = 1.0


manager = ExperimentManager(base_dir='/kaggle/working/ResNeSt50')

# Setup K-Fold
skf = stratified_kfold_split(train_data, n_splits=N_FOLDS, random_state=42)

print(f"Inizio Training su {N_FOLDS} Folds...")
print(f"Device: {Config.DEVICE}")


for fold_idx, (train_patients, val_patients) in enumerate(skf):
    print(f"\n{'='*20} FOLD {fold_idx} {'='*20}")
    

    train_loader, val_loader = create_data_loaders(
        train_patients, val_patients, 
        batch_size=Config.BATCH_SIZE, 
        num_workers=Config.NUM_WORKERS
    )
    

    fold_train_labels = []
    for p in train_patients:
        fold_train_labels.extend([s['label'] for s in p['samples']])
    
    class_weights = get_class_weights(fold_train_labels, Config.DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    print(f"Class Weights: {class_weights.cpu().numpy()}")


    model = GlobalResNeSt50(num_classes=3, pretrained=True)
    

    if hasattr(model.model, 'fc'):
        num_ftrs = model.model.fc.in_features
        model.model.fc = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(num_ftrs, 3)
        )
    model = model.to(Config.DEVICE)
    

    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=Config.LEARNING_RATE, 
        weight_decay=0.05 
    )
    
    scaler = torch.amp.GradScaler('cuda')
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=Config.NUM_EPOCHS, 
        eta_min=1e-6
    )


    best_bal_acc = 0.0
    best_val_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_bal_acc': [], 'val_bal_acc': []}
    

    for epoch in range(Config.NUM_EPOCHS):

        train_loss, train_bal_acc = train_epoch(
            model, train_loader, optimizer, criterion, Config.DEVICE, scaler,
            accumulation_steps=ACCUMULATION_STEPS, max_grad_norm=MAX_GRAD_NORM
        )

        val_loss, val_bal_acc, val_acc, val_preds, val_labels = validate(
            model, val_loader, criterion, Config.DEVICE
        )
        

        scheduler.step()
        

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_bal_acc'].append(train_bal_acc)
        history['val_bal_acc'].append(val_bal_acc)
        

        print(f"Ep {epoch+1:02d} | "
              f"T_Loss: {train_loss:.4f} T_BalAcc: {train_bal_acc:.4f} | "
              f"V_Loss: {val_loss:.4f} V_BalAcc: {val_bal_acc:.4f}")
        

        if val_bal_acc > best_bal_acc:
            best_bal_acc = val_bal_acc
            best_val_loss = val_loss
            manager.save_model(model, fold_idx, filename='best_model.pth')
            best_preds = val_preds
            best_labels = val_labels
            

    manager.save_history(history, fold_idx)
    manager.save_confusion_matrix(best_labels, best_preds, fold_idx)
    manager.log_fold_metrics(fold_idx, best_labels, best_preds, best_val_loss)
    

    del model, optimizer, scaler, scheduler
    torch.cuda.empty_cache()

Inizio Training su 5 Folds...
Device: cuda

==================== FOLD 0 ====================
Class Weights: [0.70690966 1.4235474  1.1326034 ]


model.safetensors:   0%|          | 0.00/110M [00:00<?, ?B/s]

Ep 01 | T_Loss: 1.0292 T_BalAcc: 0.4718 | V_Loss: 1.0545 V_BalAcc: 0.5219
Ep 02 | T_Loss: 0.9744 T_BalAcc: 0.5355 | V_Loss: 0.8852 V_BalAcc: 0.5967
Ep 03 | T_Loss: 0.8650 T_BalAcc: 0.6005 | V_Loss: 0.9331 V_BalAcc: 0.6263
Ep 04 | T_Loss: 0.8212 T_BalAcc: 0.6271 | V_Loss: 0.8969 V_BalAcc: 0.6138
Ep 05 | T_Loss: 0.7575 T_BalAcc: 0.6569 | V_Loss: 0.9164 V_BalAcc: 0.5754
Ep 06 | T_Loss: 0.7132 T_BalAcc: 0.6909 | V_Loss: 0.8197 V_BalAcc: 0.6816
Ep 07 | T_Loss: 0.6839 T_BalAcc: 0.7123 | V_Loss: 0.8177 V_BalAcc: 0.7358
Ep 08 | T_Loss: 0.6420 T_BalAcc: 0.7582 | V_Loss: 0.9391 V_BalAcc: 0.5796
Ep 09 | T_Loss: 0.5828 T_BalAcc: 0.7598 | V_Loss: 1.1280 V_BalAcc: 0.6478
Ep 10 | T_Loss: 0.5513 T_BalAcc: 0.7667 | V_Loss: 0.8954 V_BalAcc: 0.7422
Ep 11 | T_Loss: 0.5376 T_BalAcc: 0.7709 | V_Loss: 0.5890 V_BalAcc: 0.7646
Ep 12 | T_Loss: 0.5285 T_BalAcc: 0.7849 | V_Loss: 0.9218 V_BalAcc: 0.6747
Ep 13 | T_Loss: 0.4612 T_BalAcc: 0.8124 | V_Loss: 0.9653 V_BalAcc: 0.6866
Ep 14 | T_Loss: 0.4239 T_BalAcc: 0.832

In [23]:
manager.save_final_summary()


=== FINAL RESULTS ACROSS FOLDS ===
          fold  balanced_accuracy  accuracy  best_val_loss
mean  2.000000           0.823313  0.819886       0.791684
std   1.581139           0.021911  0.020781       0.313712


## Training Results

In [ ]:
summary_path = manager.base_dir / 'final_summary.csv'

if summary_path.exists():
    df_results = pd.read_csv(summary_path)
    
    print("\n" + "="*40)
    print(" RISULTATI FINALI CROSS-VALIDATION")
    print("="*40)
    print(df_results)
    
    avg_bal_acc = df_results['balanced_accuracy'].mean()
    std_bal_acc = df_results['balanced_accuracy'].std()
    
    print(f"\nMedia Balanced Accuracy: {avg_bal_acc:.4f} ± {std_bal_acc:.4f}")

        

    print(f"\nEsempio Confusion Matrix (Fold {N_FOLDS-1}):")
    last_cm_path = manager.base_dir / f'fold_{N_FOLDS-1}' / 'confusion_matrix.png'
    if last_cm_path.exists():
        pil_im = Image.open(last_cm_path)
        display(pil_im)
else:
    print("Nessun risultato trovato.")